# Aula 7 — Regressão Linear Supervisionada
## Modelagem, métricas de erro e análise de resíduos

Neste laboratório, construiremos um modelo para estimar o **tempo de backup (minutos)** a partir do **volume de dados (GB)**. Execute as células na ordem usando **Shift + Enter**.

## 1. Importar as bibliotecas

- `numpy`: geração de números e cálculos;
- `pandas`: organização dos dados em tabela;
- `matplotlib`: criação dos gráficos;
- `scikit-learn`: separação, treinamento e avaliação do modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Criar um conjunto de dados didático

Os valores simulam backups reais: volumes maiores tendem a exigir mais tempo, mas há variações causadas por rede, disco, servidor e concorrência de processos. O `ruído` representa essas influências não incluídas no modelo.

In [ ]:
rng = np.random.default_rng(42)
volume = rng.uniform(5, 200, 300)
ruido = rng.normal(0, 5, 300)
tempo = 2.5 + 0.32 * volume + ruido

dados = pd.DataFrame({
    "volume_gb": volume,
    "tempo_min": tempo
})

dados.head()

In [ ]:
print(dados.info())
display(dados.describe().round(2))
print("Valores ausentes por coluna:")
display(dados.isna().sum())

## 3. Visualizar a relação entre entrada e alvo

Cada ponto representa um backup. Procuramos uma tendência aproximadamente linear antes de escolher a regressão linear.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(dados["volume_gb"], dados["tempo_min"], alpha=0.65)
plt.xlabel("Volume de dados (GB)")
plt.ylabel("Tempo de backup (min)")
plt.title("Relação entre volume e tempo de backup")
plt.grid(alpha=0.2)
plt.show()

## 4. Separar treino e teste

- `X`: característica usada para explicar ou prever;
- `y`: valor numérico que queremos estimar;
- treino: dados usados para aprender os coeficientes;
- teste: dados novos, reservados para avaliar a generalização.

In [ ]:
X = dados[["volume_gb"]]
y = dados["tempo_min"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Exemplos de treino:", len(X_treino))
print("Exemplos de teste:", len(X_teste))

## 5. Criar um baseline

O baseline sempre prevê a média do conjunto de treino. O modelo linear só é útil se superar essa referência simples.

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_treino, y_treino)
previsoes_baseline = baseline.predict(X_teste)

mae_baseline = mean_absolute_error(y_teste, previsoes_baseline)
print(f"MAE do baseline: {mae_baseline:.2f} minutos")

## 6. Treinar a regressão linear

O modelo aprende uma equação no formato **ŷ = b₀ + b₁x**. O intercepto `b₀` é o ponto de partida da reta e `b₁` indica quanto a previsão muda quando o volume aumenta 1 GB.

In [ ]:
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

intercepto = modelo.intercept_
coeficiente = modelo.coef_[0]

print(f"Equação aprendida: tempo = {intercepto:.2f} + {coeficiente:.3f} × volume")
print(f"Interpretação: cada 1 GB adicional acrescenta cerca de {coeficiente:.3f} minuto(s).")

In [ ]:
novo_volume = pd.DataFrame({"volume_gb": [120]})
previsao_120 = modelo.predict(novo_volume)[0]
print(f"Tempo estimado para 120 GB: {previsao_120:.2f} minutos")

## 7. Avaliar com dados de teste

- **MAE**: erro médio em minutos;
- **MSE**: eleva os erros ao quadrado e pune mais os erros grandes;
- **RMSE**: raiz do MSE, volta à unidade minutos;
- **R²**: proporção da variação explicada pelo modelo. Não é uma medida de acurácia.

In [ ]:
previsoes = modelo.predict(X_teste)

mae = mean_absolute_error(y_teste, previsoes)
mse = mean_squared_error(y_teste, previsoes)
rmse = np.sqrt(mse)
r2 = r2_score(y_teste, previsoes)

resultados = pd.DataFrame({
    "Métrica": ["MAE", "MSE", "RMSE", "R²"],
    "Resultado": [mae, mse, rmse, r2]
})
display(resultados.round(3))
print(f"O modelo reduziu o MAE em {mae_baseline - mae:.2f} minutos em relação ao baseline.")

In [ ]:
comparacao = pd.DataFrame({
    "volume_gb": X_teste["volume_gb"].values,
    "tempo_real": y_teste.values,
    "tempo_previsto": previsoes
})
comparacao["residuo"] = comparacao["tempo_real"] - comparacao["tempo_previsto"]
display(comparacao.head(10).round(2))

## 8. Visualizar a reta e os resíduos

Um resíduo é calculado por **valor real − valor previsto**. Um bom gráfico de resíduos tende a mostrar pontos espalhados de forma aleatória ao redor de zero, sem curva ou formato de funil.

In [ ]:
ordem = np.argsort(X_teste["volume_gb"].values)

plt.figure(figsize=(9, 5))
plt.scatter(X_teste["volume_gb"], y_teste, alpha=0.65, label="Valores reais")
plt.plot(
    X_teste["volume_gb"].values[ordem],
    previsoes[ordem],
    color="crimson", linewidth=3, label="Regressão linear"
)
plt.xlabel("Volume de dados (GB)")
plt.ylabel("Tempo de backup (min)")
plt.title("Valores reais e reta de regressão")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
residuos = y_teste.values - previsoes

plt.figure(figsize=(9, 5))
plt.scatter(previsoes, residuos, alpha=0.7)
plt.axhline(0, color="crimson", linestyle="--", linewidth=2)
plt.xlabel("Tempo previsto (min)")
plt.ylabel("Resíduo: real − previsto (min)")
plt.title("Análise gráfica dos resíduos")
plt.grid(alpha=0.2)
plt.show()

## 9. Experimento: aumentar o ruído

Execute a célula e compare com os resultados anteriores. Antes de executar, formule uma hipótese: o que acontecerá com MAE, RMSE e R² quando os dados se tornarem menos previsíveis?

In [ ]:
ruido_maior = rng.normal(0, 15, 300)
tempo_com_mais_ruido = 2.5 + 0.32 * volume + ruido_maior

X2_treino, X2_teste, y2_treino, y2_teste = train_test_split(
    X, tempo_com_mais_ruido, test_size=0.25, random_state=42
)
modelo_ruidoso = LinearRegression()
modelo_ruidoso.fit(X2_treino, y2_treino)
previsoes_ruidosas = modelo_ruidoso.predict(X2_teste)

mae_ruidoso = mean_absolute_error(y2_teste, previsoes_ruidosas)
rmse_ruidoso = np.sqrt(mean_squared_error(y2_teste, previsoes_ruidosas))
r2_ruidoso = r2_score(y2_teste, previsoes_ruidosas)

comparacao_metricas = pd.DataFrame({
    "Cenário": ["Ruído original", "Ruído maior"],
    "MAE": [mae, mae_ruidoso],
    "RMSE": [rmse, rmse_ruidoso],
    "R²": [r2, r2_ruidoso]
})
display(comparacao_metricas.round(3))

## 10. Síntese para entregar

Em até cinco linhas, responda:

1. Qual equação o modelo aprendeu?
2. O modelo superou o baseline? Use uma métrica como evidência.
3. O que o MAE representa neste problema?
4. Os resíduos parecem aleatórios ou apresentam um padrão?
5. O que mudou quando o ruído aumentou?

> Conclusão esperada: um modelo não deve ser defendido apenas por gerar previsões; ele precisa superar uma referência simples, apresentar erros aceitáveis para o contexto e ter resíduos sem padrão evidente.